# Dự Án Khai Phá & Thống Kê 5.000 Phim Điện Ảnh từ Wikipedia (2015–2025)

**Môn học:** Khai phá dữ liệu & Xử lý ngôn ngữ tự nhiên
**Đề tài:** Thu thập dữ liệu 5.000 phim điện ảnh từ Wikipedia, trích xuất thực thể, mối quan hệ và thực hiện thống kê cơ bản.

---

## 1. Giới thiệu dự án
- **Nguồn dữ liệu:** Wikipedia API và Wikidata SPARQL/pageprops.
- **Đối tượng:** 5.000 phim điện ảnh phát hành trong giai đoạn 2015 – 2025.
- **Mục tiêu:**
  1. Thu thập dữ liệu văn bản và metadata infobox của 5.000 phim.
  2. Chuẩn hóa, làm sạch và loại bỏ trùng lặp.
  3. Trích xuất mạng thực thể (Movie, Person, Genre, Organization, Country, Language) và quan hệ.
  4. Thống kê mô tả và trực quan hóa toàn diện bằng biểu đồ.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.dpi': 150
})

# Xác định thư mục dữ liệu
BASE_DIR = Path("..").resolve()
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_RESULTS = BASE_DIR / "data" / "results"
DATA_RAW = BASE_DIR / "data" / "raw"

print(f"Thư mục gốc: {BASE_DIR}")

## 2. Tải dữ liệu đã qua tiền xử lý

In [ ]:
movies_file = DATA_PROCESSED / "movies.csv"
entities_file = DATA_PROCESSED / "entities.csv"
relations_file = DATA_PROCESSED / "relations.csv"
log_file = DATA_RAW / "crawl_log.csv"

df_movies = pd.read_csv(movies_file)
df_entities = pd.read_csv(entities_file)
df_relations = pd.read_csv(relations_file)
df_log = pd.read_csv(log_file) if log_file.exists() else pd.DataFrame()

print(f"✓ Movies DataFrame: {df_movies.shape[0]:,} dòng, {df_movies.shape[1]} cột")
print(f"✓ Entities DataFrame: {df_entities.shape[0]:,} thực thể")
print(f"✓ Relations DataFrame: {df_relations.shape[0]:,} quan hệ")
if not df_log.empty:
    print(f"✓ Crawl Log: {df_log.shape[0]:,} lượt crawl bài viết")

df_movies.head(3)

## 3. Thống kê & Trực quan hóa 10 tiêu chí đề tài

In [ ]:
# 1. Số lượng phim theo năm phát hành
year_counts = df_movies['release_year'].value_counts().sort_index()

plt.figure(figsize=(10, 4.5))
bars = plt.bar(year_counts.index.astype(str), year_counts.values, color='#3b82f6', edgecolor='#1d4ed8', width=0.6)
for bar in bars:
    h = bar.get_height()
    plt.annotate(f"{int(h)}", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3),
                 textcoords="offset points", ha="center", va="bottom", fontweight="bold")
plt.title("1. Số lượng phim theo năm phát hành (2015 – 2025)")
plt.xlabel("Năm phát hành")
plt.ylabel("Số lượng phim")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Top 10 Thể loại phổ biến nhất
genres_series = df_movies['genre'].dropna().apply(lambda x: [g.strip() for g in str(x).split(' ; ') if g.strip()])
all_genres = [g for sub in genres_series for g in sub]
top_genres = pd.Series(all_genres).value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index, legend=False, palette="mako")
for i, v in enumerate(top_genres.values):
    plt.text(v + (max(top_genres.values) * 0.01), i, f"{v:,}", va="center", fontweight="bold")
plt.title("2. Top 10 Thể loại phim phổ biến nhất")
plt.xlabel("Số phim")
plt.ylabel("Thể loại")
plt.tight_layout()
plt.show()

In [ ]:
# 3 & 4. Top 10 Đạo diễn và Top 10 Diễn viên
dir_series = df_movies['director'].dropna().apply(lambda x: [d.strip() for d in str(x).split(' ; ') if d.strip()])
all_dirs = [d for sub in dir_series for d in sub]
top_dirs = pd.Series(all_dirs).value_counts().head(10)

cast_series = df_movies['cast'].dropna().apply(lambda x: [a.strip() for a in str(x).split(' ; ') if a.strip()])
all_cast = [a for sub in cast_series for a in sub]
top_cast = pd.Series(all_cast).value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.barplot(x=top_dirs.values, y=top_dirs.index, hue=top_dirs.index, legend=False, ax=axes[0], palette="crest")
axes[0].set_title("3. Top 10 Đạo diễn xuất hiện nhiều nhất")
axes[0].set_xlabel("Số lượng phim")

sns.barplot(x=top_cast.values, y=top_cast.index, hue=top_cast.index, legend=False, ax=axes[1], palette="viridis")
axes[1].set_title("4. Top 10 Diễn viên xuất hiện nhiều nhất")
axes[1].set_xlabel("Số lượng phim")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Phân bố phim theo quốc gia sản xuất
country_series = df_movies['country'].dropna().apply(lambda x: [c.strip() for c in str(x).split(' ; ') if c.strip()])
all_countries = [c for sub in country_series for c in sub]
top_countries = pd.Series(all_countries).value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, hue=top_countries.index, legend=False, palette="magma")
for i, v in enumerate(top_countries.values):
    plt.text(v + (max(top_countries.values) * 0.01), i, f"{v:,}", va="center", fontweight="bold")
plt.title("5. Top 10 Quốc gia sản xuất nhiều phim nhất")
plt.xlabel("Số lượng phim")
plt.ylabel("Quốc gia")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Thời lượng phim: Trung bình, Trung vị và Phân bố
runtimes = df_movies['runtime'].dropna()
mean_val = runtimes.mean()
median_val = runtimes.median()

plt.figure(figsize=(10, 5))
sns.histplot(runtimes, kde=True, color="#059669", bins=30)
plt.axvline(mean_val, color="#dc2626", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.1f} min")
plt.axvline(median_val, color="#d97706", linestyle="-.", linewidth=2, label=f"Median: {median_val:.1f} min")
plt.title("6. Phân bố thời lượng phim (Runtime Distribution)")
plt.xlabel("Thời lượng (phút)")
plt.ylabel("Số lượng phim")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Thời lượng trung bình: {mean_val:.2f} phút | Trung vị: {median_val:.1f} phút | Độ lệch chuẩn: {runtimes.std():.2f}")

In [ ]:
# 7 & 8. Thống kê Thực thể và Quan hệ theo loại
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

e_counts = df_entities['entity_type'].value_counts()
sns.barplot(x=e_counts.values, y=e_counts.index, hue=e_counts.index, legend=False, ax=axes[0], palette="coolwarm")
axes[0].set_title("7. Số lượng thực thể theo loại (Entities)")
axes[0].set_xlabel("Số lượng")

r_counts = df_relations['relation_type'].value_counts()
sns.barplot(x=r_counts.values, y=r_counts.index, hue=r_counts.index, legend=False, ax=axes[1], palette="rocket")
axes[1].set_title("8. Số lượng quan hệ theo loại (Relations)")
axes[1].set_xlabel("Số lượng")

plt.tight_layout()
plt.show()

In [ ]:
# 9 & 10. Tỷ lệ thiếu dữ liệu và Trạng thái Crawl
missing_rates = {}
for col in df_movies.columns:
    if col == 'runtime':
        m_count = df_movies[col].isna().sum()
    else:
        m_count = ((df_movies[col] == "") | df_movies[col].isna()).sum()
    missing_rates[col] = round((m_count / len(df_movies)) * 100, 2)

sr_missing = pd.Series(missing_rates).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.barplot(x=sr_missing.values, y=sr_missing.index, hue=sr_missing.index, legend=False, ax=axes[0], palette="YlOrRd_r")
axes[0].set_title("9. Tỷ lệ thiếu dữ liệu theo trường (%)")
axes[0].set_xlabel("Tỷ lệ thiếu (%)")

if not df_log.empty and "status" in df_log.columns:
    status_counts = df_log["status"].value_counts()
else:
    status_counts = pd.Series({"SUCCESS": len(df_movies), "FAILED": 0})

colors = ["#10b981", "#ef4444"] if len(status_counts) == 2 else ["#10b981"]
axes[1].pie(status_counts.values, labels=status_counts.index, autopct="%1.1f%%", startangle=140, colors=colors)
axes[1].set_title("10. Trạng thái thu thập bài viết (Crawl Status)")

plt.tight_layout()
plt.show()